# Problem set 5 Code

In [3]:
import numpy as np 
import pandas as pd

### Problem 1(c)

In [4]:
sig = 0.036
mu = 0.018
gamma = 10
beta = 0.99

rf = -np.log(beta) + gamma * mu  -0.5 * gamma**2 * sig**2
risk_prem = gamma * sig**2

print(f'Risk Free Rate: {rf}')
print(f'Risk Premium: {risk_prem}')


Risk Free Rate: 0.12525033585350145
Risk Premium: 0.01296


Increasing $\sigma$ increases the risk premium and decreases the risk free rate. Future states are more risky so more demand for risk free which decreases the rate. Increasing $\mu$ increases the risk free rate as increasing mu means we will see increasing consumption in the future. We need a higher risk free rate to have the agent save since they expect higher consumption. Increasing $\gamma$ increases our risk aversion and thus increases the risk premium required to make the agent willing enough to invest. Similarly we are more willing to take a lower risk free rate, when $\gamma\mu < 1/2\gamma^2\sigma^2$. Otherwise the risk free rate increases since our increasing consumption dominates our aversion to risky asset. They won't need a high risk free rate to save the more risk averse they are. Increasing $\beta$ decreases the risk free rate as the agent is more patient and willing to consume less in the current period. Thus, they don't such a high risk free rate to optimally save. 

### Problem 1(d)

In [5]:
epsilon = np.random.normal(0,1,100000)

log_consumption = mu+sig*epsilon

sdf = np.log(beta) - gamma*log_consumption

rf_series = np.full(len(epsilon),rf)

log_ret = rf_series + gamma*sig**2 + sig*epsilon

log_excess_ret = log_ret - rf_series

In [6]:
print('Unconditional Standard Deviations')
print(f'Log consumption growth: {np.round(np.std(log_consumption), 4)}')
print(f'Log SDF:                {np.round(np.std(sdf), 4)}')
print(f'Log risk-free rate:     {np.round(np.std(rf_series), 4)}')
print(f'Log equity return:      {np.round(np.std(log_ret), 4)}')
print(f'Log excess return:      {np.round(np.std(log_excess_ret), 4)}')

from statsmodels.tsa.stattools import acf

ac_log_consumption = acf(log_consumption,nlags=1)[1]
ac_sdf = acf(sdf,nlags=1)[1]
ac_rf = acf(rf_series,nlags=1)[1]
ac_log_ret = acf(log_ret,nlags=1)[1]
ac_excess_ret = acf(log_excess_ret,nlags=1)[1]

print('')
print('First Autocorrelations')
print(f'Log Consumption Growth: {np.round(ac_log_consumption, 4)}')
print(f'Log SDF:                {np.round(ac_sdf, 4)}')
print(f'Risk Free Rate:         {np.round(ac_rf, 4)}')
print(f'Log Equity Return:      {np.round(ac_log_ret, 4)}')
print(f'Log Excess Return:      {np.round(ac_excess_ret, 4)}')


Unconditional Standard Deviations
Log consumption growth: 0.0361
Log SDF:                0.3612
Log risk-free rate:     0.0
Log equity return:      0.0361
Log excess return:      0.0361

First Autocorrelations
Log Consumption Growth: 0.0005
Log SDF:                0.0005
Risk Free Rate:         1.0
Log Equity Return:      0.0005
Log Excess Return:      0.0005


In [7]:
print(f'Correlation of consumption growth and equity return {np.corrcoef(log_consumption,log_ret)[0][1]}')

Correlation of consumption growth and equity return 1.0


In [8]:
print(f'expected log excess return {np.mean(log_excess_ret)}')

expected log excess return 0.012750109521684481


These statistics are consistent with the data as our standard deviations are close to the set deviation of 0.036. SDF varies by 10*0.036 which matches.  Risk free rate also is persistent and constant. The autocorrelations are low as consumption doesn't depend on previous period. Since they all depend on log consumption the autocorrelations should be low.

### Problem 2(b)

In [ ]:
np.random.seed(0)
rho = 0.4
sig = np.sqrt(0.036**2*(1-0.4**2))
mu = 0.018
gamma = 10
beta = 0.99
n=100000

log_consumption_growth = np.zeros(n)
epsilon = np.random.normal(0,1,n)
for i in range(len(log_consumption_growth)):
    if i == 0:
        log_consumption_growth[i] = (1-rho)*mu + sig*epsilon[i]
    else:
        log_consumption_growth[i] = (1-rho)*mu + rho *log_consumption_growth[i-1] + sig*epsilon[i]

sdf = np.log(beta) -gamma*log_consumption_growth
rf = -np.log(beta) + gamma*(1-rho)*mu + gamma*rho*log_consumption_growth - gamma**2*sig**2/2
rf_bar =  -np.log(beta) + gamma * mu  -0.5 * gamma**2 * sig**2

In [ ]:
from scipy.optimize import brentq

def gap(dtpt):
    rho_cs = 1/ (1+np.exp(dtpt))
    kappa = -np.log(rhos_cs) -(1-rho_cs)*np.log(1/rho_cs-1)
    risk_prem = gamma*sig**2 * 1-rho_cs*(1-gamma)*rho/(1-rho*rho_cs)
    first = (-kappa-mu+rf_bar+risk_prem)/(1-rho_cs)
    second = (1-gamma)
    return -dtpt